In [6]:
!pip install transformers datasets torch huggingface_hub -q


In [7]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import pandas as pd
import torch

In [8]:
df = pd.read_csv("mental_health_data (1).csv", header=None)
df.rename(columns={0: "text"}, inplace=True)
df = df[df["text"].notna()]
df = df[df["text"].str.strip() != ""]
df["text"] = df["text"].str.replace(r"\\n", "\n", regex=True)
df["text"] = df["text"].str.replace(r"\\t", " ", regex=True)
print(f"✅ Data loaded! Rows: {len(df)}")

✅ Data loaded! Rows: 217


In [9]:
my_data = load_dataset("csv",
    data_files="mental_health_data (1).csv")

print("Rows:", len(my_data['train']))
print("Columns:", my_data['train'].column_names)
model_name = "EleutherAI/gpt-neo-125m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded!")



Rows: 432
Columns: ["User: I feel so stressed out today\\nBot: I hear you. Stress can feel really overwhelming. Take a deep breath with me. You don't have to carry all of this alone."]


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


In [10]:
from transformers import AutoTokenizer
from datasets import Dataset

# Step 1: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Mandatory for padding

# Step 2: Convert pandas DataFrame to HuggingFace Dataset
# Assume df me single column "text" hai jisme already:
# User: ...\nBot: ...
dataset = Dataset.from_pandas(df)

# Step 3: Tokenization + label preparation
def format_data(example):
    text = example["text"] + tokenizer.eos_token  # EOS added for sequence end

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    # Set pad tokens to -100 for loss ignoring
    tokens["labels"] = [
        -100 if id == tokenizer.pad_token_id else id
        for id in tokens["input_ids"]
    ]
    return tokens

# Step 4: Apply formatting to dataset
tokenized_dataset = dataset.map(
    format_data,
    remove_columns=dataset.column_names
)

print("✅ Data formatted and ready for fine-tuning!")

Map:   0%|          | 0/217 [00:00<?, ? examples/s]

✅ Data formatted and ready for fine-tuning!


In [11]:
from huggingface_hub import notebook_login
notebook_login()

In [14]:
training_args = TrainingArguments(
    output_dir="./empathetic_bot",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    fp16=True,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    push_to_hub=False
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collator
)

# -------------------------
# TRAIN
# -------------------------
trainer.train()

print("Training complete!")


Step,Training Loss
10,1.965125
20,2.047230
30,1.903090
40,1.582596
50,1.606205


Training complete!


In [15]:
import pandas as pd

# Loss history
logs = [x for x in trainer.state.log_history if 'loss' in x]
df_logs = pd.DataFrame(logs)
print("=" * 40)
print("TRAINING COMPLETE - PROOF")
print("=" * 40)
print(f"Total examples: {len(tokenized_dataset)}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Final Loss: {trainer.state.log_history[-1]}")
print("=" * 40)
print(df_logs)

TRAINING COMPLETE - PROOF
Total examples: 217
Epochs: 2
Final Loss: {'train_runtime': 14.7554, 'train_samples_per_second': 29.413, 'train_steps_per_second': 3.795, 'total_flos': 56681940123648.0, 'train_loss': 1.7845707109996252, 'epoch': 2.0, 'step': 56}
       loss  grad_norm  learning_rate     epoch  step
0  1.965125   6.471532       0.000042  0.357143    10
1  2.047230   6.818788       0.000033  0.714286    20
2  1.903090   4.914007       0.000024  1.071429    30
3  1.582596   5.566984       0.000015  1.428571    40
4  1.606205   5.077270       0.000006  1.785714    50


In [16]:
model.save_pretrained("./empathetic_bot")
tokenizer.save_pretrained("./empathetic_bot")
print("✅ Model locally save ho gaya!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model locally save ho gaya!


In [1]:
!pip install huggingface_hub -q
from huggingface_hub import notebook_login

# Yahan HF token daalo
# Token kahan se milega:
# huggingface.co → Settings → Access Tokens → New Token (write permission)
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Apna username daalo
HF_USERNAME = "huzaifashahid"
MODEL_REPO  = f"{HF_USERNAME}/empathetic-bot"

model.push_to_hub(MODEL_REPO)
tokenizer.push_to_hub(MODEL_REPO)

print(f"✅ Model HuggingFace pe upload ho gaya!")
print(f"🔗 Link: https://huggingface.co/{MODEL_REPO}")